### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 4: Generative AI. Exercise 4**

**Attention: towards transformers and LLMs**

Instead of counting every possible context, a transformer lets each word look back at the other words in the sentence and decide how much each one matters. This is attention.
We compute attention by hand for one short sentence, add the causal mask used by language models, and check the result with PyTorch.

**The maths**

Each word is a vector of numbers (an embedding), and the sentence is a matrix $X$ with one row per word. Three weight matrices, like the ones in chapter 3, give a query $Q$ (what the word is looking for), a key $K$ (what the word offers) and a value $V$ (what the word passes on):

$$Q = X W_Q \qquad K = X W_K \qquad V = X W_V$$

The similarity between the query of one word and the key of another is a dot product. Softmax turns the similarities of each row into weights that add up to 1, and each word becomes a weighted mix of the values ($d$ is the length of the vectors):

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^\top}{\sqrt{d}} \right) V$$

A transformer repeats this attention, followed by a small neural network, many times. The last layer uses softmax over the vocabulary to predict the next word, trained with cross-entropy (chapter 2).

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import torch

# Check installation
print("All libraries imported successfully")
print(np.__version__)
print(torch.__version__)

In [ ]:
# A sentence and one embedding (vector of 3 numbers) per word
# In a real model the embeddings are learned. Here we choose them by hand
words = ["the", "tumour", "cell", "divides", "fast"]
X = np.array([
    [0.1, 0.1, 0.0],   # the
    [1.0, 0.0, 0.3],   # tumour
    [0.4, 0.2, 0.8],   # cell
    [0.2, 0.9, 0.5],   # divides
    [0.9, 0.6, 0.1]])  # fast
print("X shape (words, embedding):", X.shape)

In [ ]:
# Softmax row by row (chapter 3): each row adds up to 1
def softmax(scores):
    exps = np.exp(scores - np.max(scores, axis = 1, keepdims = True))
    return exps / np.sum(exps, axis = 1, keepdims = True)

# Plot attention weights: row = word that looks, column = word it looks at
def plot_attention(weights, title):
    plt.figure(figsize = (6, 5))
    plt.imshow(weights, cmap = 'Blues')
    plt.colorbar(label = 'attention weight')
    plt.xticks(range(len(words)), words)
    plt.yticks(range(len(words)), words)
    plt.xlabel('looks at')
    plt.ylabel('word')
    plt.title(title)
    plt.show()

In [ ]:
# Queries, keys and values: three weight matrices
# In a real model they are learned. Here they are random
np.random.seed(0)
d = 3
W_Q = np.random.normal(0, 1, size = (d, d))
W_K = np.random.normal(0, 1, size = (d, d))
W_V = np.random.normal(0, 1, size = (d, d))

Q = X @ W_Q
K = X @ W_K
V = X @ W_V
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

In [ ]:
# Step 1: similarity between every query and every key
scores = Q @ K.T / np.sqrt(d)
print("Scores (words x words):\n", np.round(scores, 2))

In [ ]:
# Step 2: softmax, so each row becomes weights that add up to 1
weights = softmax(scores)
print("Attention weights:\n", np.round(weights, 2))
print("Rows add up to:", np.round(weights.sum(axis = 1), 2))
plot_attention(weights, 'Attention weights')

In [ ]:
# Step 3: each word becomes a weighted mix of the values of all words
output = weights @ V
print("Output (words x embedding):\n", np.round(output, 2))
print("Shape:", output.shape)

In [ ]:
# Causal mask: to predict the next word, a word can only look at itself and the words before it
# We set the scores of future words to -infinity, so softmax gives them weight 0
mask = np.triu(np.ones((len(words), len(words))), k = 1) == 1
scores_masked = np.where(mask, -np.inf, scores)
weights_causal = softmax(scores_masked)
output_causal = weights_causal @ V
print("Causal attention weights:\n", np.round(weights_causal, 2))
plot_attention(weights_causal, 'Causal attention weights')

In [ ]:
# Check with PyTorch implementation
Q_torch = torch.tensor(Q, dtype = torch.float32)
K_torch = torch.tensor(K, dtype = torch.float32)
V_torch = torch.tensor(V, dtype = torch.float32)

output_torch = torch.nn.functional.scaled_dot_product_attention(Q_torch, K_torch, V_torch).numpy()
output_causal_torch = torch.nn.functional.scaled_dot_product_attention(Q_torch, K_torch, V_torch, is_causal = True).numpy()

print("Attention close: ", np.allclose(output, output_torch, atol = 1e-5))
print("Causal attention close: ", np.allclose(output_causal, output_causal_torch, atol = 1e-5))